# Jupyter Notebooks para proyectos de IA

Magic commands, display enriquecido, widgets interactivos y buenas prácticas
para usar Jupyter como entorno de experimentación con LLMs.

## 1. Magic commands más útiles

In [ ]:
# %time — medir tiempo de una celda
%time sum(range(1_000_000))

In [ ]:
# %timeit — benchmarking con múltiples ejecuciones
import json

datos = {'modelo': 'claude-haiku-4-5-20251001', 'tokens': 1024, 'temperatura': 0.7}

%timeit json.dumps(datos)
%timeit json.dumps(datos, ensure_ascii=False)

In [ ]:
# %%capture — capturar output (útil para suprimir verbosidad de instalaciones)
%%capture output_capturado
print('Este texto no se mostrará directamente')
import sys
print('Python:', sys.version)

In [ ]:
# Ver el output capturado cuando lo necesites
print('Output capturado:')
print(output_capturado.stdout)

In [ ]:
# %env — ver/establecer variables de entorno
import os

# Ver todas las variables relevantes para IA
vars_ia = {k: v[:8] + '...' if len(v) > 8 else v
           for k, v in os.environ.items()
           if any(kw in k.upper() for kw in ['API', 'KEY', 'TOKEN', 'SECRET'])}

if vars_ia:
    print('Variables de API detectadas:')
    for k, v in vars_ia.items():
        print(f'  {k}: {v}')
else:
    print('No se detectaron claves de API en el entorno')
    print('Tip: crea un archivo .env en la raíz del proyecto y usa python-dotenv')

## 2. Display enriquecido

In [ ]:
from IPython.display import display, Markdown, HTML, JSON

# Renderizar respuesta de Claude como Markdown
respuesta_simulada = """
## Análisis de la consulta

La función `calcular_coste()` tiene **dos problemas** principales:

1. No valida que `tokens_entrada` sea positivo
2. El diccionario de precios debería ser una constante, no un valor local

### Versión corregida:

```python
PRECIOS = {'haiku': (0.80, 4.00), 'sonnet': (3.00, 15.00)}

def calcular_coste(tokens_entrada: int, tokens_salida: int, modelo: str = 'haiku') -> float:
    if tokens_entrada < 0 or tokens_salida < 0:
        raise ValueError('Los tokens no pueden ser negativos')
    p = PRECIOS.get(modelo, PRECIOS['haiku'])
    return (tokens_entrada * p[0] + tokens_salida * p[1]) / 1_000_000
```
"""

display(Markdown(respuesta_simulada))

In [ ]:
# Mostrar JSON de respuesta de API de forma legible
respuesta_api = {
    'id': 'msg_01XFDUDYJgAACzvnptvVoYEL',
    'type': 'message',
    'role': 'assistant',
    'content': [{'type': 'text', 'text': 'Hola, ¿en qué puedo ayudarte?'}],
    'model': 'claude-haiku-4-5-20251001',
    'stop_reason': 'end_turn',
    'usage': {'input_tokens': 25, 'output_tokens': 12},
}

display(JSON(respuesta_api))

In [ ]:
# HTML para tablas comparativas
tabla_html = """
<table style='border-collapse: collapse; width: 100%; font-family: monospace;'>
  <tr style='background: #2196F3; color: white;'>
    <th style='padding: 8px; text-align: left;'>Modelo</th>
    <th style='padding: 8px; text-align: right;'>Entrada ($/1M)</th>
    <th style='padding: 8px; text-align: right;'>Salida ($/1M)</th>
    <th style='padding: 8px; text-align: center;'>Uso recomendado</th>
  </tr>
  <tr style='background: #f5f5f5;'>
    <td style='padding: 8px;'>claude-haiku-4-5</td>
    <td style='padding: 8px; text-align: right;'>$0.80</td>
    <td style='padding: 8px; text-align: right;'>$4.00</td>
    <td style='padding: 8px; text-align: center;'>Clasificación, resúmenes simples</td>
  </tr>
  <tr>
    <td style='padding: 8px;'>claude-sonnet-4-6</td>
    <td style='padding: 8px; text-align: right;'>$3.00</td>
    <td style='padding: 8px; text-align: right;'>$15.00</td>
    <td style='padding: 8px; text-align: center;'>Razonamiento complejo, código</td>
  </tr>
</table>
"""

display(HTML(tabla_html))

## 3. Widgets interactivos para experimentar con parámetros

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Calculadora interactiva de costes
w_tokens_entrada = widgets.IntSlider(value=500, min=0, max=10000, step=100,
                                      description='Tokens entrada:', style={'description_width': 'initial'})
w_tokens_salida = widgets.IntSlider(value=200, min=0, max=4096, step=50,
                                     description='Tokens salida:', style={'description_width': 'initial'})
w_modelo = widgets.Dropdown(options=['haiku', 'sonnet'], value='haiku', description='Modelo:')
w_llamadas = widgets.IntSlider(value=1000, min=1, max=100000, step=1000,
                                description='Llamadas/mes:', style={'description_width': 'initial'})

output = widgets.Output()

PRECIOS = {'haiku': (0.80, 4.00), 'sonnet': (3.00, 15.00)}

def actualizar_coste(change):
    with output:
        clear_output(wait=True)
        p = PRECIOS[w_modelo.value]
        coste_por_llamada = (w_tokens_entrada.value * p[0] + w_tokens_salida.value * p[1]) / 1_000_000
        coste_mensual = coste_por_llamada * w_llamadas.value
        print(f'Coste por llamada: ${coste_por_llamada:.6f}')
        print(f'Coste mensual ({w_llamadas.value:,} llamadas): ${coste_mensual:.2f}')
        print(f'Coste anual estimado: ${coste_mensual * 12:.2f}')

for widget in [w_tokens_entrada, w_tokens_salida, w_modelo, w_llamadas]:
    widget.observe(actualizar_coste, names='value')

display(widgets.VBox([w_modelo, w_tokens_entrada, w_tokens_salida, w_llamadas, output]))
actualizar_coste(None)

## 4. Buenas prácticas para notebooks de IA

In [ ]:
# PRÁCTICA 1: Siempre aislar la configuración en las primeras celdas
import os
from pathlib import Path

# Configuración centralizada — fácil de cambiar sin buscar en todo el notebook
MODELO = os.getenv('MODELO_IA', 'claude-haiku-4-5-20251001')
MAX_TOKENS = int(os.getenv('MAX_TOKENS', '1024'))
TEMPERATURA = float(os.getenv('TEMPERATURA', '0.3'))
DIR_DATOS = Path(os.getenv('DIR_DATOS', '/tmp/datos_ia'))
DIR_DATOS.mkdir(exist_ok=True)

print(f'Configuración activa:')
print(f'  Modelo: {MODELO}')
print(f'  Max tokens: {MAX_TOKENS}')
print(f'  Temperatura: {TEMPERATURA}')
print(f'  Directorio datos: {DIR_DATOS}')

In [ ]:
# PRÁCTICA 2: Guardar resultados intermedios para no repetir llamadas costosas
import json
from pathlib import Path
from datetime import datetime

def guardar_resultado(nombre: str, datos: dict | list, directorio: Path = DIR_DATOS) -> Path:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    archivo = directorio / f'{nombre}_{timestamp}.json'
    with open(archivo, 'w', encoding='utf-8') as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    print(f'Guardado: {archivo}')
    return archivo

def cargar_ultimo_resultado(nombre: str, directorio: Path = DIR_DATOS) -> dict | list | None:
    archivos = sorted(directorio.glob(f'{nombre}_*.json'))
    if not archivos:
        return None
    with open(archivos[-1]) as f:
        return json.load(f)

# Uso
resultado_ejemplo = {'respuestas': ['Resp 1', 'Resp 2'], 'modelo': MODELO, 'coste': 0.0023}
archivo = guardar_resultado('analisis_sentimiento', resultado_ejemplo)

# En la siguiente celda, antes de llamar a la API, intentar cargar
cached = cargar_ultimo_resultado('analisis_sentimiento')
if cached:
    print(f'Usando resultado en caché: {cached}')
else:
    print('Haciendo nueva llamada a la API...')

In [ ]:
# PRÁCTICA 3: Reinicio limpio con verificación de estado
def verificar_entorno() -> dict:
    import sys
    estado = {
        'python': sys.version.split()[0],
        'api_key_ok': bool(os.getenv('ANTHROPIC_API_KEY')),
        'librerias': {},
    }
    for lib in ['anthropic', 'numpy', 'pandas', 'httpx']:
        try:
            m = __import__(lib)
            estado['librerias'][lib] = getattr(m, '__version__', 'ok')
        except ImportError:
            estado['librerias'][lib] = 'NO INSTALADA'
    return estado

estado = verificar_entorno()
print(f'Python {estado["python"]}')
print(f'API Key: {"✅" if estado["api_key_ok"] else "❌ Falta ANTHROPIC_API_KEY"}')
print('Librerías:')
for lib, version in estado['librerias'].items():
    ok = version != 'NO INSTALADA'
    print(f'  {"✅" if ok else "❌"} {lib}: {version}')